# Final evaluation v2: test_v2, opened once

`configs/final_v2.yaml`. Step 1 repeats the whole pipeline on validation data and checks every run against its
reported numbers; **test_v2 is not opened unless this passes.** Step 2 evaluates test_v2 once. Step 3 adds 95%
bootstrap intervals from the saved test_v2 predictions (CPU, no new prediction).

**Before running:**
1. **Input**: `dataset-ai-pictionary` (processed data), the output of `ai-pictionary-build-test-v2` (test_v2), and the
   output of `ai-pictionary-checkpoints` (every checkpoint and validation prediction). Nothing else is needed.
2. **GPU T4 x2**, **Internet on**. **Save Version → Save & Run All**. Expected: 4-6 hours.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping")


In [ ]:
packages = sorted({p.parent for p in Path("/kaggle/input").rglob("val_predictions") if p.is_dir() and (p.parent / "runs").is_dir()})
if len(packages) != 1:
    raise RuntimeError(f"expected one checkpoint package (attach the output of ai-pictionary-checkpoints), found {packages}")
PACKAGE = packages[0]
VAL_PREDICTIONS, RUNS = PACKAGE / "val_predictions", PACKAGE / "runs"
folders = sorted({p.parent for p in Path("/kaggle/input").rglob("test_v2_meta.parquet")})
if len(folders) != 1:
    raise RuntimeError(f"expected one test_v2 folder (attach the output of ai-pictionary-build-test-v2), found {folders}")
TEST_V2 = folders[0]
print("package:", PACKAGE, len(list(RUNS.iterdir())), "runs,", len(list(VAL_PREDICTIONS.glob("*.npz"))), "validation predictions")
print("test_v2:", TEST_V2)


In [ ]:
# Step 1: validation check. test_v2 is not opened unless this passes.
done = subprocess.run(f"PYTHONPATH=src python -m quickdraw_exp.final_eval --matrix configs/final_v2.yaml --split val "
                      f"--val-predictions {VAL_PREDICTIONS} --runs {RUNS}", shell=True, capture_output=True, text=True)
print(done.stdout[-6000:])
print(done.stderr[-3000:])
if done.returncode != 0:
    raise RuntimeError("validation check failed: test_v2 was NOT opened")


In [ ]:
# Step 2: the one test_v2 run.
!PYTHONPATH=src python -m quickdraw_exp.final_eval --matrix configs/final_v2.yaml --split test_v2 --test-v2 {TEST_V2} --val-predictions {VAL_PREDICTIONS} --runs {RUNS}


In [ ]:
# Step 3: bootstrap intervals from the saved test_v2 predictions (CPU).
!PYTHONPATH=src python -m quickdraw_exp.bootstrap --matrix configs/final_v2.yaml --report /kaggle/working/runs/final_v2_test_v2/results.json --val-predictions {VAL_PREDICTIONS} --final-predictions /kaggle/working/runs/final_v2_test_v2


In [ ]:
import json

check = json.loads(Path("/kaggle/working/runs/final_v2_val/results.json").read_text())["check"]
print("validation check:", check["compared"], "numbers, largest difference", check["largest"],
      "probs max abs diff", round(check["probs_max_abs_diff"], 5))
r = json.loads(Path("/kaggle/working/runs/final_v2_test_v2/results.json").read_text())

def show(rows, label):
    print(f"\n{label}")
    for row in rows:
        tag = f"[{row['precision_target']}] " if "precision_target" in row else ""
        if row.get("difference") is None:
            print(f"  {tag}{row['name']}: not available")
        else:
            print(f"  {tag}{row['name']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, difference {row['difference']:+.4f}  "
                  f"(a {[round(v, 4) for v in row['a_values']]}, b {[round(v, 4) for v in row['b_values']]})")

for name, s in r["sets"].items():
    print(f"\n==================== {name}: {s['drawings']} drawings, {s['sampled_drawings']} sampled ====================")
    print("Top-1 by ink fraction:")
    for run, by in s["leak"]["per_run"].items():
        print(f"  {run:24s}", {f: round(v["top1"], 4) for f, v in by.items()})
    show(s["leak"]["comparisons"], "Early top-1 (10, 25, 50% ink):")
    show(s["anytime"]["comparisons"], "When to guess (AUC):")
    print("\nPer run: AUC (pmax / conf), risk control at 2% and 5%:")
    for run, rep in s["anytime"]["per_run"].items():
        rc = rep["pmax"]["risk_control"]
        conf = f" conf {rep['conf']['auc']:.4f}" if "conf" in rep else ""
        print(f"  {run:24s} pmax {rep['pmax']['auc']:.4f}{conf}  gap2% {rc['0.02']['gap']:.4f} met {rc['0.02']['met']} "
              f"earliness {rc['0.02']['earliness']:.3f}  gap5% {rc['0.05']['gap']:.4f} met {rc['0.05']['met']} "
              f"earliness {rc['0.05']['earliness']:.3f}  end {rep['reference']['end_of_drawing']['accuracy']:.4f}")
    show(s["hierarchy"]["comparisons"], "Hierarchy (reward area):")
